# AIC 2026 — Component 02  
# BEiT3 Adaptive Keyframe PTS Shadow — v2.3.0

## DECISION

**Notebook 01 v2.2.3 is accepted. Do not rerun it.**

Component 02 consumes its existing executed output.

This notebook is a **shadow adaptive-semantic keyframe experiment**. It does not
replace active Stage03/SigLIP2 and it does not modify Online Master.

---

## INPUTS

### INPUT A — executed Notebook 01 output

Source notebook:

`01_AUTOSHOT_PTS_STREAMING_v2_2_3.ipynb`

Accepted executed build:

`aic2026-autoshot-pts-streaming-v2.2.3`

Required artifacts:

```text
component01_handoff.json
per_video/<video_id>.json
boundary_npz/<video_id>.npz
```

The accepted run contains:

```text
873 / 873 videos
0 errors
cuda:0 + cuda:1 participation
checkpoint coverage = 1.0 on both GPUs
actual-PTS round-trip = PASS
```

Attach the **existing Notebook Output** of the successful `1rr-note` run.  
Do not run Notebook 01 again.

### INPUT B — raw corpus

`nguyenminhtric/data-batch-1-aic`

Used only to decode the exact videos referenced by Notebook 01.

### INPUT C — BEiT3 checkpoint

Canonical legacy source:

```text
repo     = Quintu/beit3
filename = beit3_large_patch16_384_coco_retrieval.pth
```

Resolution order:

1. `AIC2026_BEIT3_CKPT=/exact/local/path.pth`
2. shallow search of mounted Kaggle inputs for the exact filename
3. Hugging Face download if `AIC2026_BEIT3_ALLOW_DOWNLOAD=1` (default)

Checkpoint SHA-256 and state-dict coverage are recorded.

---

## WHAT IS REUSED

- AutoShot scenes and actual PTS from executed Notebook 01.
- Raw Batch1 videos.
- Legacy BEiT3 COCO-retrieval checkpoint/architecture.

## WHAT IS REBUILT

Only the **BEiT3 shadow adaptive keyframe proposal layer**.

## WHAT IS NOT CHANGED

- Stage03 remains active KIS.
- SigLIP2 remains active visual backbone.
- AutoShot 01 output remains frozen.
- No canonical ShotRef is created.
- No accuracy claim is made without GT/proxy evidence.

---

## v2.3.0 RUNTIME CHANGES

The previous v2.2 design would sample every **0.32 s**. On the actual accepted
Notebook-01 scene registry (99,402 scenes, 129.58 h), that means
approximately **1,602,546 BEiT3 candidate embeddings**.

v2.3.0 defaults to **1.0 s** candidate spacing:

```text
0.32 s → 1,602,546 candidates
1.00 s → 613,302 candidates
```

This is about **2.61× fewer** candidate embeddings.

The legacy 0.32 s density is still available by setting:

`AIC2026_BEIT3_STEP_SEC=0.32`

but it is no longer the runtime-safe default.

Other changes:

- removes duplicate legacy config that silently overrode batch settings;
- consumes exact executed 01 lineage instead of `/kaggle/input/autoshot-v2`;
- rebinds raw videos through `source_relpath`, never stale absolute paths;
- CPU decodes candidate frames directly to 384×384 before GPU transfer;
- adaptive BEiT3 batch with same-GPU OOM backoff;
- no second full-video decode to save WebP images by default;
- duration/candidate-aware two-GPU balancing;
- correct wall-clock two-GPU throughput metrics;
- exact PTS only; no `round(timestamp × FPS)`.

In [1]:
# ── CELL 01: dependencies, immutable constants, strict 2-GPU contract ─────────
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import os, sys, json, hashlib, math, time, shutil, zipfile, importlib.util, subprocess
import numpy as np
import torch

def ensure_pkg(import_name,pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable,"-m","pip","install","--quiet",pip_name or import_name]
        )

for import_name,pip_name in (
    ("av","av"),
    ("faiss","faiss-cpu"),
    ("torchscale","torchscale"),
    ("timm","timm"),
    ("PIL","pillow"),
    ("huggingface_hub","huggingface_hub"),
    ("psutil","psutil"),
):
    ensure_pkg(import_name,pip_name)

import av
import faiss
import psutil
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from timm.models.layers import trunc_normal_ as trunc_normal_fn
from huggingface_hub import hf_hub_download

BUILD_VERSION="aic2026-beit3-adaptive-keyframe-pts-v2.3.0"
STAGE_ID="component02_beit3_adaptive_keyframe_shadow"

EXPECTED_01_BUILD="aic2026-autoshot-pts-streaming-v2.2.3"
EXPECTED_01_VIDEO_COUNT=873
EXPECTED_01_CHECKPOINT_SHA256="3e85290546ce6d32f4a3581ec2cae87aedd2402246a0d46b4d361a330b4b1fa6"

RAW_DATASET_HANDLE="nguyenminhtric/data-batch-1-aic"
BEIT3_REPO="Quintu/beit3"
BEIT3_CKPT_FILENAME="beit3_large_patch16_384_coco_retrieval.pth"

IMG_SIZE=384
CANDIDATE_STEP_SEC=float(os.environ.get("AIC2026_BEIT3_STEP_SEC","1.0"))
MIN_KF_GAP_SEC=float(os.environ.get("AIC2026_BEIT3_MIN_KF_GAP_SEC","1.6"))
THRESH_KF=float(os.environ.get("AIC2026_BEIT3_THRESH_KF","0.95"))
THRESH_PREV=float(os.environ.get("AIC2026_BEIT3_THRESH_PREV","0.95"))
BATCH_SIZE=int(os.environ.get("AIC2026_BEIT3_BATCH","16"))
MAX_CANDIDATES=int(os.environ.get("AIC2026_BEIT3_MAX_CANDIDATES","800000"))
ALLOW_HF_DOWNLOAD=os.environ.get("AIC2026_BEIT3_ALLOW_DOWNLOAD","1")=="1"
SAVE_SELECTED_IMAGES=os.environ.get("AIC2026_BEIT3_SAVE_IMAGES","0")=="1"
FORCE_REBUILD=os.environ.get("AIC2026_FORCE_REBUILD","0")=="1"

if CANDIDATE_STEP_SEC<=0:
    raise ValueError(("CANDIDATE_STEP_SEC",CANDIDATE_STEP_SEC))
if MIN_KF_GAP_SEC<=0:
    raise ValueError(("MIN_KF_GAP_SEC",MIN_KF_GAP_SEC))
if not (0.0<THRESH_KF<=1.0 and 0.0<THRESH_PREV<=1.0):
    raise ValueError(("SIM_THRESHOLDS",THRESH_KF,THRESH_PREV))
if BATCH_SIZE<1:
    raise ValueError(("BATCH_SIZE",BATCH_SIZE))

if not torch.cuda.is_available() or torch.cuda.device_count()<2:
    raise RuntimeError(("STRICT_DUAL_GPU_REQUIRED",torch.cuda.device_count()))
GPU_DEVICES=("cuda:0","cuda:1")

BASE=Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK_ROOT=BASE/"aic2026_vnext"/"component02_beit3_shadow_v230"
IMAGE_ROOT=WORK_ROOT/"images"
MATERIALIZED=WORK_ROOT/"materialized_inputs"
for p in (WORK_ROOT,IMAGE_ROOT,MATERIALIZED):
    p.mkdir(parents=True,exist_ok=True)

def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("[PASS] strict dual GPU preflight",GPU_DEVICES)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 15.2 MB/s eta 0:00:00
[PASS] strict dual GPU preflight ('cuda:0', 'cuda:1')


In [2]:
# ── CELL 02: bounded resolution of executed Notebook-01 output + raw corpus ───
def resolve_raw_dataset():
    explicit=os.environ.get("AIC2026_VIDEO_ROOT","").strip()
    if explicit:
        p=Path(explicit).resolve()
        if not p.is_dir():
            raise RuntimeError(("AIC2026_VIDEO_ROOT_NOT_DIRECTORY",str(p)))
        return p,"EXPLICIT_ENV"

    candidates=[
        Path("/kaggle/input/data-batch-1-aic"),
        Path("/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic"),
    ]
    hits=sorted({p.resolve() for p in candidates if p.is_dir()},key=str)
    if not hits:
        raise RuntimeError({
            "message":"RAW_AIC2026_INPUT_REQUIRED",
            "dataset_handle":RAW_DATASET_HANDLE,
        })
    return hits[0],"EXACT_KAGGLE_HANDLE"

RAW_ROOT,RAW_ROOT_SOURCE=resolve_raw_dataset()

def valid_01_handoff(path):
    try:
        d=json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None
    if d.get("build_version")!=EXPECTED_01_BUILD:
        return None
    return d

def inspect_zip_for_01(zp):
    try:
        with zipfile.ZipFile(zp,"r") as z:
            for info in z.infolist():
                if Path(info.filename).name!="component01_handoff.json":
                    continue
                try:
                    d=json.loads(z.read(info).decode("utf-8"))
                except Exception:
                    continue
                if d.get("build_version")==EXPECTED_01_BUILD:
                    return info.filename
    except zipfile.BadZipFile:
        return None
    return None

def materialize_01_zip(zp):
    dest=MATERIALIZED/"component01"
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zp,"r") as z:
        bad=z.testzip()
        if bad is not None:
            raise RuntimeError(("ZIP_CRC_FAILED",str(zp),bad))
        for info in z.infolist():
            rel=Path(info.filename)
            if rel.is_absolute() or ".." in rel.parts:
                raise RuntimeError(("ZIP_PATH_ESCAPE",info.filename))
            target=(dest/rel).resolve()
            if dest.resolve()!=target and dest.resolve() not in target.parents:
                raise RuntimeError(("ZIP_PATH_ESCAPE",info.filename))
            if info.is_dir():
                target.mkdir(parents=True,exist_ok=True)
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(info) as fi,target.open("wb") as fo:
                shutil.copyfileobj(fi,fo)
    hits=sorted(dest.rglob("component01_handoff.json"),key=str)
    hits=[p.resolve() for p in hits if valid_01_handoff(p) is not None]
    if len(hits)!=1:
        raise RuntimeError(("COMPONENT01_ZIP_HANDOFF_COUNT",len(hits),[str(x) for x in hits]))
    return hits[0]

def resolve_component01():
    explicit_root=os.environ.get("AIC2026_AUTOSHOT01_ROOT","").strip()
    if explicit_root:
        root=Path(explicit_root).resolve()
        hits=sorted(root.rglob("component01_handoff.json"),key=str)
        hits=[p.resolve() for p in hits if valid_01_handoff(p) is not None]
        if len(hits)!=1:
            raise RuntimeError(("AIC2026_AUTOSHOT01_ROOT_INVALID",str(root),[str(x) for x in hits]))
        return hits[0],"EXPLICIT_ROOT"

    explicit_zip=os.environ.get("AIC2026_AUTOSHOT01_ZIP","").strip()
    if explicit_zip:
        zp=Path(explicit_zip).resolve()
        if inspect_zip_for_01(zp) is None:
            raise RuntimeError(("AIC2026_AUTOSHOT01_ZIP_INVALID",str(zp)))
        return materialize_01_zip(zp),"EXPLICIT_ZIP"

    # Exact expected notebook-output mount first.
    exact_candidates=[
        Path("/kaggle/input/notebooks/nguyenminhtric/1rr-note/aic2026_vnext/component01_autoshot/component01_handoff.json"),
        Path("/kaggle/input/1rr-note/aic2026_vnext/component01_autoshot/component01_handoff.json"),
    ]
    exact=[p.resolve() for p in exact_candidates if p.is_file() and valid_01_handoff(p) is not None]
    if len(exact)==1:
        return exact[0],"EXACT_1rr_NOTEBOOK_OUTPUT"

    root=Path("/kaggle/input")
    patterns=(
        "*/aic2026_vnext/component01_autoshot/component01_handoff.json",
        "notebooks/*/*/aic2026_vnext/component01_autoshot/component01_handoff.json",
        "datasets/*/*/aic2026_vnext/component01_autoshot/component01_handoff.json",
        "*/component01_handoff.json",
    )
    hits=[]
    if root.is_dir():
        for pat in patterns:
            hits.extend(p.resolve() for p in root.glob(pat) if p.is_file())
    hits=sorted({p for p in hits if valid_01_handoff(p) is not None},key=str)
    if len(hits)==1:
        return hits[0],"BOUNDED_MOUNT_PATTERN"
    if len(hits)>1:
        raise RuntimeError(("MULTIPLE_COMPONENT01_HANDOFFS",[str(x) for x in hits]))

    zip_hits=[]
    if root.is_dir():
        for pat in ("*.zip","*/*.zip","notebooks/*/*/*.zip","datasets/*/*/*.zip"):
            for zp in root.glob(pat):
                if not zp.is_file():
                    continue
                member=inspect_zip_for_01(zp)
                if member is not None:
                    zip_hits.append((zp.resolve(),member))
    zip_hits=sorted(set(zip_hits),key=lambda x:(str(x[0]),x[1]))
    if len(zip_hits)==1:
        return materialize_01_zip(zip_hits[0][0]),"BOUNDED_ZIP"
    if zip_hits:
        raise RuntimeError(("MULTIPLE_COMPONENT01_ZIPS",[(str(z),m) for z,m in zip_hits]))

    raise RuntimeError({
        "message":"EXECUTED_AUTOSHOT01_v2_2_3_OUTPUT_REQUIRED",
        "required_build":EXPECTED_01_BUILD,
        "action":"Kaggle Add Input -> Notebook Output -> successful 1rr-note run; do NOT rerun Notebook 01.",
    })

H01_PATH,H01_SOURCE=resolve_component01()
H01=json.loads(H01_PATH.read_text(encoding="utf-8"))
H01_ROOT=H01_PATH.parent

if H01.get("build_version")!=EXPECTED_01_BUILD:
    raise RuntimeError(("COMPONENT01_BUILD_DRIFT",H01.get("build_version")))
if H01.get("checkpoint_sha256")!=EXPECTED_01_CHECKPOINT_SHA256:
    raise RuntimeError(("COMPONENT01_CHECKPOINT_DRIFT",H01.get("checkpoint_sha256")))

print("[PASS] frozen parents",{
    "component01_source":H01_SOURCE,
    "component01_handoff":str(H01_PATH),
    "component01_build":H01.get("build_version"),
    "raw_root":str(RAW_ROOT),
    "raw_source":RAW_ROOT_SOURCE,
})

[PASS] frozen parents {'component01_source': 'EXACT_1rr_NOTEBOOK_OUTPUT', 'component01_handoff': '/kaggle/input/notebooks/nguyenminhtric/1rr-note/aic2026_vnext/component01_autoshot/component01_handoff.json', 'component01_build': 'aic2026-autoshot-pts-streaming-v2.2.3', 'raw_root': '/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic', 'raw_source': 'EXACT_KAGGLE_HANDLE'}


In [3]:
# ── CELL 03: resolve and validate all 873 AutoShot per-video scene files ───────
def locate_per_video_root():
    candidates=[
        Path(str(H01.get("per_video_root",""))),
        H01_ROOT/"per_video",
        H01_ROOT.parent/"per_video",
    ]
    for p in candidates:
        if p.is_dir():
            return p.resolve()
    roots=sorted({
        p.parent.resolve()
        for p in H01_ROOT.rglob("*.json")
        if p.parent.name=="per_video"
    },key=str)
    if len(roots)!=1:
        raise RuntimeError(("COMPONENT01_PER_VIDEO_ROOT_NOT_FOUND",[
            str(x) for x in roots
        ]))
    return roots[0]

PER_VIDEO_ROOT=locate_per_video_root()
JSONS=sorted(PER_VIDEO_ROOT.glob("*.json"),key=lambda p:p.name)
if len(JSONS)!=EXPECTED_01_VIDEO_COUNT:
    raise RuntimeError(("COMPONENT01_PER_VIDEO_COUNT",len(JSONS),EXPECTED_01_VIDEO_COUNT))

jobs=[]
total_scenes=0
for jp in JSONS:
    d=json.loads(jp.read_text(encoding="utf-8"))
    if d.get("build_version")!=EXPECTED_01_BUILD:
        raise RuntimeError(("PER_VIDEO_BUILD_DRIFT",str(jp),d.get("build_version")))
    if d.get("time_truth")!="ACTUAL_DECODED_PTS":
        raise RuntimeError(("PER_VIDEO_TIME_TRUTH",str(jp),d.get("time_truth")))
    if d.get("pts_monotonic") is not True:
        raise RuntimeError(("PER_VIDEO_PTS_NOT_MONOTONIC",str(jp)))

    video_id=str(d["video_id"])
    rel=str(d["source_relpath"])
    video_path=(RAW_ROOT/rel).resolve()
    if RAW_ROOT.resolve()!=video_path and RAW_ROOT.resolve() not in video_path.parents:
        raise RuntimeError(("RAW_PATH_ESCAPE",video_id,rel))
    if not video_path.is_file():
        raise FileNotFoundError(("RAW_VIDEO_MISSING",video_id,str(video_path)))
    if video_path.stem!=video_id:
        raise RuntimeError(("VIDEO_ID_RAW_PATH_MISMATCH",video_id,video_path.stem))

    scenes=d.get("scenes",[])
    for si,s in enumerate(scenes):
        for k in ("start_pts_time","end_pts_time","start_pts_raw","end_pts_raw"):
            if k not in s:
                raise RuntimeError(("ACTUAL_PTS_SCENE_FIELD_REQUIRED",video_id,si,k))
        if float(s["end_pts_time"])<float(s["start_pts_time"]):
            raise RuntimeError(("SCENE_INTERVAL_INVALID",video_id,si))

    jobs.append({
        "json_path":str(jp),
        "video_id":video_id,
        "source_relpath":rel,
        "video_path":str(video_path),
        "scenes":scenes,
        "frame_count":int(d["frame_count"]),
        "pts_time_max":float(d["pts_time_max"]),
    })
    total_scenes += len(scenes)

if len({j["video_id"] for j in jobs})!=EXPECTED_01_VIDEO_COUNT:
    raise RuntimeError("COMPONENT01_VIDEO_ID_COLLISION")

print("[PASS] Component01 scene registry",{
    "videos":len(jobs),
    "scenes":total_scenes,
    "per_video_root":str(PER_VIDEO_ROOT),
})

[PASS] Component01 scene registry {'videos': 873, 'scenes': 99402, 'per_video_root': '/kaggle/input/notebooks/nguyenminhtric/1rr-note/aic2026_vnext/component01_autoshot/per_video'}


In [4]:
# ── CELL 04: candidate-time policy, pre-run budget, balanced dual-GPU plan ─────
def target_times_for_scene(scene):
    a=float(scene["start_pts_time"])
    b=float(scene["end_pts_time"])
    span=max(0.0,b-a)
    nbase=int(math.floor(span/CANDIDATE_STEP_SEC))+1
    xs=[a+i*CANDIDATE_STEP_SEC for i in range(nbase)]
    if not xs or abs(xs[-1]-b)>1e-6:
        xs.append(b)
    return xs

def estimated_targets(job):
    return sum(len(target_times_for_scene(s)) for s in job["scenes"])

for j in jobs:
    j["estimated_targets"]=estimated_targets(j)

TOTAL_TARGETS=int(sum(j["estimated_targets"] for j in jobs))
if TOTAL_TARGETS>MAX_CANDIDATES:
    raise RuntimeError({
        "message":"BEIT3_CANDIDATE_BUDGET_EXCEEDED",
        "estimated_candidates":TOTAL_TARGETS,
        "max_candidates":MAX_CANDIDATES,
        "candidate_step_sec":CANDIDATE_STEP_SEC,
        "action":"increase AIC2026_BEIT3_STEP_SEC or explicitly raise AIC2026_BEIT3_MAX_CANDIDATES",
    })

# Greedy longest-processing-time partition by estimated candidate count.
bins=[[],[]]
loads=[0,0]
for j in sorted(jobs,key=lambda x:(-x["estimated_targets"],x["video_id"])):
    g=0 if loads[0]<=loads[1] else 1
    bins[g].append(j)
    loads[g]+=j["estimated_targets"]

imbalance=abs(loads[0]-loads[1])/max(1,max(loads))
print("[PASS] BEiT3 workload plan",{
    "candidate_step_sec":CANDIDATE_STEP_SEC,
    "estimated_candidates":TOTAL_TARGETS,
    "gpu0_videos":len(bins[0]),
    "gpu1_videos":len(bins[1]),
    "gpu0_estimated_candidates":loads[0],
    "gpu1_estimated_candidates":loads[1],
    "estimated_imbalance_fraction":imbalance,
})

[PASS] BEiT3 workload plan {'candidate_step_sec': 1.0, 'estimated_candidates': 613302, 'gpu0_videos': 436, 'gpu1_videos': 437, 'gpu0_estimated_candidates': 306634, 'gpu1_estimated_candidates': 306668, 'estimated_imbalance_fraction': 0.00011086908317789923}


In [5]:
class BEiT3ForRetrieval(nn.Module):
    def __init__(self, args):
        super().__init__()
        from torchscale.model.BEiT3 import BEiT3
        self.beit3 = BEiT3(args)
        d = args.encoder_embed_dim
        self.vision_head = nn.Linear(d, d, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_fn(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.no_grad()
    def forward(self, x):
        out = self.beit3(textual_tokens=None, visual_tokens=x, text_padding_position=None)
        v = self.vision_head(out["encoder_out"][:, 0])
        return F.normalize(v, dim=-1)

In [6]:
# ── CELL 06: BEiT3 checkpoint resolver + strict coverage gate ─────────────────
def resolve_beit3_checkpoint():
    explicit=os.environ.get("AIC2026_BEIT3_CKPT","").strip()
    if explicit:
        p=Path(explicit).resolve()
        if not p.is_file():
            raise RuntimeError(("AIC2026_BEIT3_CKPT_MISSING",str(p)))
        return p,"EXPLICIT_ENV"

    root=Path("/kaggle/input")
    hits=[]
    if root.is_dir():
        patterns=(
            f"*/{BEIT3_CKPT_FILENAME}",
            f"*/*/{BEIT3_CKPT_FILENAME}",
            f"datasets/*/*/{BEIT3_CKPT_FILENAME}",
            f"datasets/*/*/*/{BEIT3_CKPT_FILENAME}",
            f"notebooks/*/*/{BEIT3_CKPT_FILENAME}",
        )
        for pat in patterns:
            hits.extend(p.resolve() for p in root.glob(pat) if p.is_file())
    hits=sorted(set(hits),key=str)
    if len(hits)==1:
        return hits[0],"MOUNTED_EXACT_FILENAME"
    if len(hits)>1:
        raise RuntimeError(("BEIT3_CHECKPOINT_AMBIGUOUS",[str(x) for x in hits]))

    if not ALLOW_HF_DOWNLOAD:
        raise RuntimeError({
            "message":"BEIT3_CHECKPOINT_REQUIRED",
            "repo":BEIT3_REPO,
            "filename":BEIT3_CKPT_FILENAME,
            "action":"attach checkpoint or set AIC2026_BEIT3_ALLOW_DOWNLOAD=1",
        })

    p=Path(hf_hub_download(repo_id=BEIT3_REPO,filename=BEIT3_CKPT_FILENAME)).resolve()
    return p,"HF_HUB_DOWNLOAD"

BEIT3_CKPT_PATH,BEIT3_CKPT_SOURCE=resolve_beit3_checkpoint()
BEIT3_CKPT_SHA256=sha256_file(BEIT3_CKPT_PATH)

def load_model_v230(device):
    from torchscale.architecture.config import EncoderConfig
    args=EncoderConfig(
        img_size=IMG_SIZE,
        patch_size=16,
        vocab_size=64010,
        multiway=True,
        layernorm_embedding=False,
        normalize_output=True,
        no_output_layer=True,
        drop_path_rate=0.0,
        encoder_embed_dim=1024,
        encoder_attention_heads=16,
        encoder_ffn_embed_dim=4096,
        encoder_layers=24,
        checkpoint_activations=None,
    )
    model=BEiT3ForRetrieval(args)
    raw=torch.load(BEIT3_CKPT_PATH,map_location="cpu",weights_only=True)
    state=raw.get("model",raw)
    target=model.state_dict()

    usable={}
    mismatch=[]
    for k,v in state.items():
        if k in target:
            if tuple(v.shape)==tuple(target[k].shape):
                usable[k]=v
            else:
                mismatch.append((k,tuple(v.shape),tuple(target[k].shape)))

    coverage=len(usable)/max(1,len(target))
    if mismatch or coverage<0.97:
        raise RuntimeError({
            "message":"BEIT3_CHECKPOINT_COVERAGE_FAILED",
            "coverage":coverage,
            "shape_mismatch_preview":mismatch[:10],
            "missing_preview":sorted(set(target)-set(usable))[:20],
        })

    model.load_state_dict(usable,strict=False)
    model=model.to(device).eval()
    mean=torch.tensor(IMAGENET_DEFAULT_MEAN,device=device).view(1,3,1,1)
    std=torch.tensor(IMAGENET_DEFAULT_STD,device=device).view(1,3,1,1)
    print("[PASS] BEiT3 checkpoint",device,{
        "coverage":coverage,
        "sha256":BEIT3_CKPT_SHA256,
        "source":BEIT3_CKPT_SOURCE,
    })
    return model,mean,std,coverage

print("[PASS] BEiT3 checkpoint resolved",{
    "path":str(BEIT3_CKPT_PATH),
    "source":BEIT3_CKPT_SOURCE,
    "sha256":BEIT3_CKPT_SHA256,
})

beit3_large_patch16_384_coco_retrieval.p(…):   0%|          | 0.00/1.35G [00:00<?, ?B/s]

[PASS] BEiT3 checkpoint resolved {'path': '/root/.cache/huggingface/hub/models--Quintu--beit3/blobs/72a569be8a4c5396d82f7a4eab9dd3c9d6f11af731e9d86227450cde02a2fddb', 'source': 'HF_HUB_DOWNLOAD', 'sha256': '72a569be8a4c5396d82f7a4eab9dd3c9d6f11af731e9d86227450cde02a2fddb'}


In [7]:
# ── CELL 07: actual-PTS decode + candidate embedding with adaptive GPU batch ───
def embed_rgb_batches(model,mean,std,device,rgbs,initial_batch):
    if not rgbs:
        return np.empty((0,1024),dtype=np.float32),initial_batch,0

    out=[]
    pos=0
    effective=max(1,int(initial_batch))
    forward_calls=0
    while pos<len(rgbs):
        take=min(effective,len(rgbs)-pos)
        try:
            arr=np.stack(rgbs[pos:pos+take],axis=0)
            x=torch.from_numpy(arr).permute(0,3,1,2).contiguous().float()
            x=x.to(device,non_blocking=True)/255.0
            x=(x-mean)/std
            with torch.no_grad(),torch.amp.autocast("cuda",dtype=torch.float16):
                v=model(x)
            out.append(v.float().cpu().numpy())
            pos += take
            forward_calls += 1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if take<=1:
                raise
            effective=max(1,take//2)
            print("[WARN]",device,"BEiT3 OOM backoff batch ->",effective,flush=True)

    E=np.concatenate(out,axis=0).astype(np.float32)
    E/=np.linalg.norm(E,axis=1,keepdims=True).clip(min=1e-12)
    return E,effective,forward_calls

def decode_candidate_frames(job):
    # One target record per AutoShot scene target.
    targets=[]
    for si,scene in enumerate(job["scenes"]):
        for t in target_times_for_scene(scene):
            targets.append((float(t),int(si)))
    targets.sort(key=lambda x:(x[0],x[1]))
    if not targets:
        return [],[]

    recs=[]
    rgbs=[]
    ti=0
    decode_ordinal=0

    with av.open(job["video_path"]) as container:
        stream=container.streams.video[0]
        tb=stream.time_base
        tb_num=int(tb.numerator)
        tb_den=int(tb.denominator)
        tb_float=float(tb)
        stream_start_raw=None if stream.start_time is None else int(stream.start_time)
        stream_start_sec=None if stream.start_time is None else float(stream.start_time)*tb_float

        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            pts_raw=int(frame.pts)
            pts_time=float(pts_raw)*tb_float

            due=[]
            while ti<len(targets) and pts_time+1e-9>=targets[ti][0]:
                due.append(targets[ti])
                ti += 1

            if due:
                # Resize before GPU transfer. Existing v2.2 decoded full-resolution RGB
                # then resized on GPU, which wastes CPU RAM and PCIe bandwidth.
                rgb=frame.reformat(width=IMG_SIZE,height=IMG_SIZE,format="rgb24").to_ndarray()
                for target,scene_idx in due:
                    recs.append({
                        "video_id":job["video_id"],
                        "source_relpath":job["source_relpath"],
                        "video_path":job["video_path"],
                        "scene_index":int(scene_idx),
                        "target_pts_time":float(target),
                        "actual_pts_time":float(pts_time),
                        "pts_raw":pts_raw,
                        "time_base_num":tb_num,
                        "time_base_den":tb_den,
                        "stream_start_time_raw":stream_start_raw,
                        "stream_start_time_sec":stream_start_sec,
                        "decode_ordinal":int(decode_ordinal),
                    })
                    # Duplicate references are intentional when one presentation frame
                    # satisfies targets belonging to adjacent scenes; final proposals
                    # are deduplicated by actual PTS after scene-wise selection.
                    rgbs.append(rgb)

            decode_ordinal += 1
            if ti>=len(targets):
                break

    if ti<len(targets):
        raise RuntimeError(("TARGETS_BEYOND_DECODED_VIDEO",job["video_id"],ti,len(targets)))

    return recs,rgbs

In [8]:
# ── CELL 08: semantic selection + exact actual-PTS proposal identity ───────────
def select_keyframes(recs,E):
    selected=[]
    by_scene={}
    for i,r in enumerate(recs):
        by_scene.setdefault(int(r["scene_index"]),[]).append(i)

    for scene_idx,idxs in sorted(by_scene.items()):
        if not idxs:
            continue
        chosen=[idxs[0]]
        for pos,j in enumerate(idxs[1:],start=1):
            last=chosen[-1]
            prev=idxs[pos-1]
            dt=float(recs[j]["actual_pts_time"]-recs[last]["actual_pts_time"])
            sim_last=float(np.dot(E[j],E[last]))
            sim_prev=float(np.dot(E[j],E[prev]))
            if dt>=MIN_KF_GAP_SEC or (sim_last<THRESH_KF and sim_prev<THRESH_PREV):
                chosen.append(j)
        selected.extend(chosen)
    return selected

def frame_ref_proposal_id(video_id,pts_raw,tb_num,tb_den):
    raw=f"{video_id}|{int(pts_raw)}|{int(tb_num)}/{int(tb_den)}".encode("utf-8")
    return "frp_"+hashlib.sha256(raw).hexdigest()[:24]

def dedupe_selected(recs,E,selected):
    # Same decoded presentation frame may serve adjacent AutoShot scenes.
    # Keep one proposal/vector and aggregate scene membership / target provenance.
    by_key={}
    order=[]
    for i in selected:
        r=recs[i]
        key=(r["video_id"],int(r["pts_raw"]),int(r["time_base_num"]),int(r["time_base_den"]))
        if key not in by_key:
            q=dict(r)
            q["scene_indices"]=[int(r["scene_index"])]
            q["target_pts_times"]=[float(r["target_pts_time"])]
            q["selected_vector"]=E[i].astype(np.float32)
            by_key[key]=q
            order.append(key)
        else:
            q=by_key[key]
            si=int(r["scene_index"])
            if si not in q["scene_indices"]:
                q["scene_indices"].append(si)
            q["target_pts_times"].append(float(r["target_pts_time"]))

    rows=[]
    vecs=[]
    for key in order:
        q=by_key[key]
        v=q.pop("selected_vector")
        q["scene_indices"]=sorted(q["scene_indices"])
        q["target_pts_times"]=sorted(q["target_pts_times"])
        q["target_actual_pts_abs_error_sec"]=float(
            min(abs(float(q["actual_pts_time"])-t) for t in q["target_pts_times"])
        )
        q["frame_ref_proposal_id"]=frame_ref_proposal_id(
            q["video_id"],q["pts_raw"],q["time_base_num"],q["time_base_den"]
        )
        q["canonical_status"]="ADAPTIVE_PROPOSAL_SHADOW_ONLY"
        q["active_kis_policy"]="DO_NOT_REPLACE_STAGE03_SIGLIP2"
        q["image_path"]=None
        q["image_materialization_status"]="NOT_SAVED_DEFAULT_RECAPTURE_FROM_RAW_BY_ACTUAL_PTS"
        rows.append(q)
        vecs.append(v)
    return rows,vecs

In [9]:
# ── CELL 09: per-video worker; single decode pass by default ───────────────────
def process_video(job,state):
    device=state["device"]
    t0=time.perf_counter()
    torch.cuda.reset_peak_memory_stats(torch.device(device))

    t_decode=time.perf_counter()
    recs,rgbs=decode_candidate_frames(job)
    decode_sec=float(time.perf_counter()-t_decode)

    t_embed=time.perf_counter()
    E,effective_batch,forward_calls=embed_rgb_batches(
        state["model"],state["mean"],state["std"],device,rgbs,state["batch_size"]
    )
    embed_sec=float(time.perf_counter()-t_embed)
    state["batch_size"]=effective_batch

    if len(recs)!=len(E):
        raise RuntimeError(("CANDIDATE_EMBED_LENGTH_MISMATCH",job["video_id"],len(recs),len(E)))

    selected=select_keyframes(recs,E)
    rows,vecs=dedupe_selected(recs,E,selected)

    # Optional images are intentionally not materialized by default. If explicitly
    # requested, save selected 384x384 candidate tensors without re-decoding video.
    if SAVE_SELECTED_IMAGES and rows:
        rgb_by_pts={}
        for r,rgb in zip(recs,rgbs):
            rgb_by_pts.setdefault(int(r["pts_raw"]),rgb)
        for row in rows:
            rgb=rgb_by_pts[int(row["pts_raw"])]
            outdir=IMAGE_ROOT/row["video_id"]
            outdir.mkdir(parents=True,exist_ok=True)
            out=outdir/f'{row["frame_ref_proposal_id"]}.webp'
            Image.fromarray(rgb).save(out,"WEBP",quality=90)
            row["image_path"]=str(out)
            row["image_materialization_status"]="SAVED_FROM_SINGLE_PASS_384_CANDIDATE"

    elapsed=float(time.perf_counter()-t0)
    peak=int(torch.cuda.max_memory_allocated(torch.device(device)))
    for row in rows:
        row["gpu"]=device
        row["video_elapsed_sec"]=elapsed
        row["video_decode_sec"]=decode_sec
        row["video_embed_sec"]=embed_sec
        row["video_candidate_count"]=len(recs)
        row["video_selected_count"]=len(rows)
        row["effective_batch_size"]=int(effective_batch)
        row["video_forward_calls"]=int(forward_calls)
        row["cuda_peak_bytes"]=peak

    return {
        "video_id":job["video_id"],
        "gpu":device,
        "rows":rows,
        "vecs":vecs,
        "elapsed_sec":elapsed,
        "decode_sec":decode_sec,
        "embed_sec":embed_sec,
        "candidate_count":len(recs),
        "selected_count":len(rows),
        "effective_batch_size":int(effective_batch),
        "forward_calls":int(forward_calls),
        "cuda_peak_bytes":peak,
    }

In [10]:
# ── CELL 10: strict two-GPU execution + wall-clock runtime evidence ────────────
def gpu_worker(gpu_idx,assigned):
    device=f"cuda:{gpu_idx}"
    torch.cuda.set_device(gpu_idx)
    model,mean,std,coverage=load_model_v230(device)
    state={
        "model":model,
        "mean":mean,
        "std":std,
        "device":device,
        "batch_size":BATCH_SIZE,
        "coverage":coverage,
    }
    out=[]
    for n,job in enumerate(assigned,1):
        try:
            result=process_video(job,state)
            out.append(result)
            if n==1 or n%25==0 or n==len(assigned):
                print(
                    f"[{device}] {n}/{len(assigned)} "
                    f"video={job['video_id']} "
                    f"cand={result['candidate_count']} "
                    f"sel={result['selected_count']} "
                    f"batch={result['effective_batch_size']} "
                    f"elapsed={result['elapsed_sec']:.1f}s",
                    flush=True,
                )
        except Exception as e:
            out.append({
                "video_id":job["video_id"],
                "gpu":device,
                "status":"ERROR",
                "error":repr(e),
            })
    return out

parallel_t0=time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as ex:
    f0=ex.submit(gpu_worker,0,bins[0])
    f1=ex.submit(gpu_worker,1,bins[1])
    results=f0.result()+f1.result()
parallel_wall_sec=float(time.perf_counter()-parallel_t0)

errors=[r for r in results if r.get("status")=="ERROR"]
ok=[r for r in results if r.get("status")!="ERROR"]

if errors:
    raise RuntimeError(("BEIT3_PARTIAL_FAILURE",len(errors),errors[:10]))
if len(ok)!=EXPECTED_01_VIDEO_COUNT:
    raise RuntimeError(("BEIT3_VIDEO_COUNT_MISMATCH",len(ok),EXPECTED_01_VIDEO_COUNT))

used={r["gpu"] for r in ok}
if not {"cuda:0","cuda:1"}.issubset(used):
    raise RuntimeError(("DUAL_GPU_PARTICIPATION_NOT_PROVEN",used))

rows=[]
vecs=[]
for r in ok:
    rows.extend(r["rows"])
    vecs.extend(r["vecs"])
if not rows:
    raise RuntimeError("NO_BEIT3_KEYFRAME_PROPOSALS")

print("[PASS] dual-GPU BEiT3 corpus run",{
    "videos":len(ok),
    "proposals":len(rows),
    "parallel_wall_sec":parallel_wall_sec,
    "gpu_participation":sorted(used),
})

[PASS] BEiT3 checkpoint cuda:0 {'coverage': 1.0, 'sha256': '72a569be8a4c5396d82f7a4eab9dd3c9d6f11af731e9d86227450cde02a2fddb', 'source': 'HF_HUB_DOWNLOAD'}
[PASS] BEiT3 checkpoint cuda:1 {'coverage': 1.0, 'sha256': '72a569be8a4c5396d82f7a4eab9dd3c9d6f11af731e9d86227450cde02a2fddb', 'source': 'HF_HUB_DOWNLOAD'}
[cuda:1] 1/437 video=L25_V054 cand=2704 sel=1409 batch=16 elapsed=289.1s
[cuda:0] 1/436 video=L25_V007 cand=2966 sel=1535 batch=16 elapsed=333.0s
[cuda:0] 25/436 video=L25_V076 cand=1719 sel=908 batch=16 elapsed=198.8s
[cuda:1] 25/437 video=L22_V003 cand=1731 sel=931 batch=16 elapsed=196.8s
[cuda:0] 50/436 video=L21_V016 cand=1496 sel=791 batch=16 elapsed=161.5s
[cuda:1] 50/437 video=L28_V006 cand=1500 sel=868 batch=16 elapsed=200.2s
[cuda:0] 75/436 video=L25_V021 cand=1379 sel=740 batch=16 elapsed=153.6s
[cuda:1] 75/437 video=L29_V017 cand=1379 sel=753 batch=16 elapsed=194.0s
[cuda:0] 100/436 video=L27_V007 cand=884 sel=497 batch=16 elapsed=111.9s
[cuda:1] 100/437 video=L27_V008

In [11]:
# ── CELL 11: write shadow embeddings/index/registry + runtime report ───────────
E=np.stack(vecs).astype(np.float32)
E/=np.linalg.norm(E,axis=1,keepdims=True).clip(min=1e-12)

emb_path=WORK_ROOT/"adaptive_beit3_embeddings.fp16.npy"
np.save(emb_path,E.astype(np.float16))

index=faiss.IndexFlatIP(E.shape[1])
index.add(E)
index_path=WORK_ROOT/"adaptive_beit3_shadow.index"
faiss.write_index(index,str(index_path))

registry_path=WORK_ROOT/"adaptive_keyframe_registry.jsonl"
with registry_path.open("w",encoding="utf-8") as f:
    for row in rows:
        row["embedding_row"]=int(row.get("embedding_row",-1))
    for i,row in enumerate(rows):
        row["embedding_row"]=i
        f.write(json.dumps(row,ensure_ascii=False)+"\n")

drift=np.asarray([r["target_actual_pts_abs_error_sec"] for r in rows],dtype=float)
candidate_total=int(sum(r["candidate_count"] for r in ok))
selected_total=len(rows)

per_gpu={}
for gpu in sorted({r["gpu"] for r in ok}):
    g=[r for r in ok if r["gpu"]==gpu]
    per_gpu[gpu]={
        "videos":len(g),
        "candidate_count":int(sum(r["candidate_count"] for r in g)),
        "selected_count":int(sum(r["selected_count"] for r in g)),
        "sum_video_elapsed_sec":float(sum(r["elapsed_sec"] for r in g)),
        "sum_decode_sec":float(sum(r["decode_sec"] for r in g)),
        "sum_embed_sec":float(sum(r["embed_sec"] for r in g)),
        "peak_cuda_bytes":int(max(r["cuda_peak_bytes"] for r in g)),
        "final_effective_batch_size":int(g[-1]["effective_batch_size"]),
    }

report={
    "build_version":BUILD_VERSION,
    "stage_id":STAGE_ID,
    "input_lineage":{
        "component01_build":H01.get("build_version"),
        "component01_handoff_sha256":sha256_file(H01_PATH),
        "component01_checkpoint_sha256":H01.get("checkpoint_sha256"),
        "raw_dataset_handle":RAW_DATASET_HANDLE,
        "beit3_repo":BEIT3_REPO,
        "beit3_checkpoint_filename":BEIT3_CKPT_FILENAME,
        "beit3_checkpoint_source":BEIT3_CKPT_SOURCE,
        "beit3_checkpoint_sha256":BEIT3_CKPT_SHA256,
    },
    "videos":len(ok),
    "autoshot_scenes":total_scenes,
    "candidate_step_sec":CANDIDATE_STEP_SEC,
    "candidate_count":candidate_total,
    "selected_proposals":selected_total,
    "selection_fraction":float(selected_total/max(1,candidate_total)),
    "embedding_dim":int(E.shape[1]),
    "gpu_participation":sorted(used),
    "per_gpu":per_gpu,
    "time_semantics":"ACTUAL_DECODED_PTS",
    "fps_arithmetic_used":False,
    "target_actual_pts_drift_mae_sec":float(drift.mean()),
    "target_actual_pts_drift_p95_sec":float(np.quantile(drift,0.95)),
    "target_actual_pts_drift_p99_sec":float(np.quantile(drift,0.99)),
    "parallel_wall_sec":parallel_wall_sec,
    "combined_video_per_hour":float(len(ok)*3600.0/max(1e-9,parallel_wall_sec)),
    "combined_candidate_per_sec":float(candidate_total/max(1e-9,parallel_wall_sec)),
    "combined_selected_per_sec":float(selected_total/max(1e-9,parallel_wall_sec)),
    "peak_cuda_bytes":int(max(r["cuda_peak_bytes"] for r in ok)),
    "save_selected_images":SAVE_SELECTED_IMAGES,
    "canonical_status":"SHADOW_ONLY_NOT_PROMOTED",
    "active_kis_policy":"DO_NOT_REPLACE_STAGE03_SIGLIP2",
    "accuracy_claim":"NONE_NO_GT",
    "artifacts":{
        "embedding_path":str(emb_path),
        "embedding_sha256":sha256_file(emb_path),
        "faiss_index_path":str(index_path),
        "faiss_index_sha256":sha256_file(index_path),
        "registry_path":str(registry_path),
        "registry_sha256":sha256_file(registry_path),
    },
    "validation_state":{
        "STATIC_CONTRACT_VALIDATION":"PASS",
        "CORPUS_RUNTIME_VALIDATION":"PASS",
        "ACCURACY_VALIDATION":"NOT_RUN",
        "LATENCY_VALIDATION":"MEASURED_CORPUS_RUNTIME_NOT_PROMOTION_GATE",
    },
}

report_path=WORK_ROOT/"run_report.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")

handoff={
    "schema_version":"component-handoff-v1",
    "build_version":BUILD_VERSION,
    "stage_id":STAGE_ID,
    "parent_component01_build":H01.get("build_version"),
    "parent_component01_handoff_sha256":sha256_file(H01_PATH),
    "raw_dataset_handle":RAW_DATASET_HANDLE,
    "beit3_checkpoint_sha256":BEIT3_CKPT_SHA256,
    "adaptive_embedding_path":str(emb_path),
    "adaptive_index_path":str(index_path),
    "adaptive_registry_path":str(registry_path),
    "run_report_path":str(report_path),
    "canonical_status":"SHADOW_ONLY_NOT_PROMOTED",
    "active_kis_policy":"DO_NOT_REPLACE_STAGE03_SIGLIP2",
    "next_consumer":"paired evidence benchmark / later shadow ablation",
}
handoff_path=WORK_ROOT/"component02_handoff.json"
handoff_path.write_text(json.dumps(handoff,ensure_ascii=False,indent=2),encoding="utf-8")

print(json.dumps(report,ensure_ascii=False,indent=2))
print("[DONE]",handoff_path)

{
  "build_version": "aic2026-beit3-adaptive-keyframe-pts-v2.3.0",
  "stage_id": "component02_beit3_adaptive_keyframe_shadow",
  "input_lineage": {
    "component01_build": "aic2026-autoshot-pts-streaming-v2.2.3",
    "component01_handoff_sha256": "bc3395e1b10eb528d397c7ad33a6a01c54b58229f983753db9c98b3548c2d684",
    "component01_checkpoint_sha256": "3e85290546ce6d32f4a3581ec2cae87aedd2402246a0d46b4d361a330b4b1fa6",
    "raw_dataset_handle": "nguyenminhtric/data-batch-1-aic",
    "beit3_repo": "Quintu/beit3",
    "beit3_checkpoint_filename": "beit3_large_patch16_384_coco_retrieval.pth",
    "beit3_checkpoint_source": "HF_HUB_DOWNLOAD",
    "beit3_checkpoint_sha256": "72a569be8a4c5396d82f7a4eab9dd3c9d6f11af731e9d86227450cde02a2fddb"
  },
  "videos": 873,
  "autoshot_scenes": 99402,
  "candidate_step_sec": 1.0,
  "candidate_count": 613302,
  "selected_proposals": 338461,
  "selection_fraction": 0.5518667801507251,
  "embedding_dim": 1024,
  "gpu_participation": [
    "cuda:0",
    "cuda